# Why Deep Learning Fails Across MRI Vendors
## Interactive Figure Generation from JSON Results

This notebook regenerates all key figures from the paper's JSON result files.
No GPU needed — just the JSON files in `results/`.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

ckpt1 = json.load(open('../results/checkpoint.json'))
ckpt2 = json.load(open('../results/checkpoint_v2.json'))
cka_full = json.load(open('../results/cka_full.json'))
print('Loaded all result files')

## Figure 1: B₀ Dose-Response (Non-Monotonic)

In [ ]:
b0 = ckpt2['results']['b0_dose_response']
hz = [0, 25, 50, 75, 100, 125, 150]
ds3 = [b0[f'b0_{h}hz']['ds3'] for h in hz]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hz, ds3, 'o-', color='#1f77b4', linewidth=2, markersize=8)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('B₀ Offset (Hz)', fontsize=13)
ax.set_ylabel('DS3 (error increase factor)', fontsize=13)
ax.set_title('B₀ Dose-Response: Worst at 25 Hz, Not 150 Hz', fontsize=15)
ax.grid(True, alpha=0.3)
ax.annotate('Worst damage\n(25 Hz)', xy=(25, ds3[1]), xytext=(60, ds3[1]+0.1),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=11, color='red')
plt.tight_layout()
plt.show()

## Figure 4: Algorithm Comparison

In [ ]:
seeds = [42, 123, 456]
methods = ['ERM', 'CORAL', 'GroupDRO', 'IRM', 'Mixup']
src_vals, ph_vals, ge_vals = [], [], []

# ERM, CORAL from checkpoint
for algo in ['erm', 'coral']:
    runs = [ckpt1['results'][f'algo_resnet1d_18_{algo}_seed{s}'] for s in seeds]
    src_vals.append(np.mean([r['source']['mae'] for r in runs]))
    ph_vals.append(np.mean([r['philips']['mae'] for r in runs]))
    ge_vals.append(np.mean([r['ge']['mae'] for r in runs]))

# GroupDRO
grp = ckpt2['results']['groupdro']
src_vals.append(np.mean([r['src'] for r in grp]))
ph_vals.append(np.mean([r['ph'] for r in grp]))
ge_vals.append(np.mean([r['ge'] for r in grp]))

# IRM
irm = ckpt2['results']['irm']
src_vals.append(np.mean([r['src'] for r in irm]))
ph_vals.append(np.mean([r['ph'] for r in irm]))
ge_vals.append(np.mean([r['ge'] for r in irm]))

# Mixup from enhancements
enh = json.load(open('../results/enhancements.json'))
src_vals.append(np.mean([r['src']['mae'] for r in enh['mixup']]))
ph_vals.append(np.mean([r['ph']['mae'] for r in enh['mixup']]))
ge_vals.append(np.mean([r['ge']['mae'] for r in enh['mixup']]))

x = np.arange(len(methods))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, src_vals, width, label='Source (Siemens)', color='#1f77b4')
ax.bar(x, ph_vals, width, label='Philips', color='#ff7f0e')
ax.bar(x + width, ge_vals, width, label='GE', color='#2ca02c')
ax.set_ylabel('MAE (ms)', fontsize=13)
ax.set_title('All Methods Fail on Unseen Vendors', fontsize=15)
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend()
ax.set_yscale('log')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Key Numbers Summary

In [ ]:
print('=' * 60)
print('KEY PAPER NUMBERS')
print('=' * 60)
print()
print('ERM ResNet:  src=4.3±0.7  ph=240.3±2.7  DS3=58±10')
print('Mixup ResNet: src=11.4±2.0 ph=287.7±6.2  DS3=26±5')
print('Gated Hybrid: ph=193.2 ms (14% better than ERM)')
print()
print('Bootstrap 95% CI for ERM DS3: [48.4, 72.2]')
print('ERM vs Mixup DS3: p=0.030 (significant)')
print('ERM vs CORAL DS3: p=0.302 (not significant)')
print('ERM vs GroupDRO DS3: p=0.525 (not significant)')
print()
print('CKA Within-Siemens: 0.0017 ± 0.0007')
print('CKA Siemens↔Philips: 0.0012 ± 0.0005')
print('CKA Random: 0.0004 ± 0.0001')